# Notebook 02: Gap and Decision
## Goal: CanonicalPacket → DecisionState

This notebook transforms a `CanonicalPacket` (from Notebook 01) into a `DecisionState` by:

1. **Contradiction Detection** — Find conflicting values across facts/hypotheses
2. **Blocker Identification** — Check hard/soft blockers per stage MVB
3. **Confidence Scoring** — Authority-weighted overall confidence
4. **Decision State Machine** — Exactly one of 5 output states
5. **Follow-Up Question Generation** — Ordered, templated questions

**Contract:** see `notebooks/02_gap_and_decision_contract.md`

In [ ]:
import json
from typing import List, Dict, Any, Optional, Literal, Tuple
from dataclasses import dataclass, asdict, field
from datetime import datetime
from enum import IntEnum
import uuid

# =============================================================================
# SECTION 0: Imports from Notebook 01 (CanonicalPacket, Slot, etc.)
# =============================================================================
# In production these come from a shared module.
# Re-declaring minimal versions here so the notebook is self-contained.

class AuthorityLevel(IntEnum):
    MANUAL_OVERRIDE = 1
    EXPLICIT_USER = 2
    IMPORTED_STRUCTURED = 3
    EXPLICIT_OWNER = 4
    DERIVED_SIGNAL = 5
    SOFT_HYPOTHESIS = 6
    UNKNOWN = 7

AUTHORITY_WEIGHTS = {
    "manual_override": 1.0,
    "explicit_user": 0.95,
    "imported_structured": 0.85,
    "explicit_owner": 0.80,
    "derived_signal": 0.60,
    "soft_hypothesis": 0.0,
    "unknown": 0.0,
}

def is_fact(authority_level: str) -> bool:
    return authority_level.lower() in (
        "manual_override", "explicit_user", "imported_structured", "explicit_owner"
    )

def is_derived_signal(authority_level: str) -> bool:
    return authority_level.lower() == "derived_signal"

def is_hypothesis(authority_level: str) -> bool:
    return authority_level.lower() in ("soft_hypothesis",)

@dataclass
class EvidenceRef:
    ref_id: str
    envelope_id: str
    evidence_type: str
    excerpt: str
    field_path: Optional[str] = None
    offset: Optional[Dict[str, int]] = None
    confidence: float = 1.0
    metadata: Dict[str, Any] = field(default_factory=dict)

@dataclass
class Slot:
    value: Any = None
    confidence: float = 0.0
    authority_level: str = "unknown"
    extraction_mode: str = "unknown"
    evidence_refs: List[EvidenceRef] = field(default_factory=list)
    updated_at: str = field(default_factory=lambda: datetime.now().isoformat())
    notes: Optional[str] = None

@dataclass
class UnknownField:
    field_name: str
    reason: str
    attempted_at: Optional[str] = None
    notes: Optional[str] = None

@dataclass
class CanonicalPacket:
    """Minimal version — the full version is in Notebook 01."""
    packet_id: str
    created_at: str
    last_updated: str
    facts: Dict[str, Slot] = field(default_factory=dict)
    derived_signals: Dict[str, Slot] = field(default_factory=dict)
    hypotheses: Dict[str, Slot] = field(default_factory=dict)
    unknowns: List[UnknownField] = field(default_factory=list)
    contradictions: List[Dict[str, Any]] = field(default_factory=list)
    source_envelope_ids: List[str] = field(default_factory=list)
    stage: str = "discovery"

    def all_slots(self) -> Dict[str, Slot]:
        """All known slots across all layers."""
        return {**self.facts, **self.derived_signals, **self.hypotheses}


## MVB (Minimum Viable Blockers) by Stage

Per the contract, each stage has hard blockers (must-have) and soft blockers (nice-to-have). Only **facts** and **derived signals** can resolve hard blockers — hypotheses cannot.

In [ ]:
# =============================================================================
# SECTION 1: MVB DEFINITIONS (Contract Source of Truth)
# =============================================================================

MVB_BY_STAGE = {
    "discovery": {
        "hard_blockers": [
            "destination_city",   # Where are they going?
            "origin_city",        # Where are they starting?
            "travel_dates",       # When?
            "traveler_count",     # How many people?
        ],
        "soft_blockers": [
            "budget_range",
            "trip_purpose",
            "traveler_preferences",
        ],
    },
    "shortlist": {
        "hard_blockers": [
            "destination_city",
            "origin_city",
            "travel_dates",
            "traveler_count",
            "selected_destinations",  # Narrowed down options
        ],
        "soft_blockers": [
            "activity_preferences",
            "accommodation_type",
        ],
    },
    "proposal": {
        "hard_blockers": [
            "destination_city",
            "origin_city",
            "travel_dates",
            "traveler_count",
            "selected_destinations",
            "selected_itinerary",  # Chosen from shortlist
        ],
        "soft_blockers": [
            "special_requests",
            "dietary_restrictions",
        ],
    },
    "booking": {
        "hard_blockers": [
            "destination_city",
            "origin_city",
            "travel_dates",
            "traveler_count",
            "selected_destinations",
            "selected_itinerary",
            "traveler_details",   # Names, DOB, passport info
            "payment_method",
        ],
        "soft_blockers": [],
    },
}

# Field alias map: canonical field → accepted alternate names in packet
FIELD_ALIASES = {
    "destination_city": ["destination", "dest_city"],
    "origin_city": ["origin", "departure_city", "departure"],
    "travel_dates": ["travel_dates_from", "travel_dates_to", "date_window", "dates"],
    "traveler_count": ["num_travelers", "group_size"],
    "budget_range": ["budget", "budget_total", "budget_band"],
    "trip_purpose": ["purpose", "intent", "travel_intent", "trip_intent"],
    "traveler_preferences": ["preferences", "traveler_prefs"],
    "selected_destinations": ["destinations", "shortlist"],
    "selected_itinerary": ["itinerary", "chosen_itinerary"],
    "activity_preferences": ["activities", "activity_prefs"],
    "accommodation_type": ["hotel_type", "accommodation", "lodging"],
    "special_requests": ["requests", "special_needs"],
    "dietary_restrictions": ["diet", "food_restrictions", "meal_restrictions"],
    "traveler_details": ["passenger_details", "traveler_info", "passenger_info"],
    "payment_method": ["payment", "payment_info"],
}

def resolve_field(packet: CanonicalPacket, canonical_field: str) -> Optional[Slot]:
    """
    Look up a canonical field in the packet.
    Checks: facts → derived_signals → hypotheses (in order).
    Also checks aliases.
    """
    # Direct match in facts
    if canonical_field in packet.facts:
        return packet.facts[canonical_field]
    # Direct match in derived signals
    if canonical_field in packet.derived_signals:
        return packet.derived_signals[canonical_field]
    # Direct match in hypotheses
    if canonical_field in packet.hypotheses:
        return packet.hypotheses[canonical_field]

    # Alias match — check each layer
    for aliases in FIELD_ALIASES.get(canonical_field, []):
        if aliases in packet.facts:
            return packet.facts[aliases]
        if aliases in packet.derived_signals:
            return packet.derived_signals[aliases]
        if aliases in packet.hypotheses:
            return packet.hypotheses[aliases]

    return None

def field_fills_blocker(slot: Optional[Slot]) -> bool:
    """
    Does this slot resolve a hard blocker?
    Rules:
    - Hypotheses do NOT fill blockers
    - Facts and derived signals DO fill blockers
    - Unknown authority does NOT fill blockers
    """
    if slot is None:
        return False
    if is_hypothesis(slot.authority_level):
        return False
    if slot.authority_level.lower() == "unknown":
        return False
    return True


## Contradiction Policy

Defines how conflicting values are handled. Each contradiction type maps to an action.

In [ ]:
# =============================================================================
# SECTION 2: CONTRADICTION POLICY
# =============================================================================

CONTRADICTION_HANDLING = {
    "destination_conflict": {
        "action": "ASK_FOLLOWUP",
        "priority": "critical",
        "message": "Multiple destinations mentioned. Please clarify."
    },
    "date_conflict": {
        "action": "STOP_NEEDS_REVIEW",
        "priority": "critical",
        "message": "Conflicting travel dates detected."
    },
    "budget_conflict": {
        "action": "BRANCH_OPTIONS",
        "priority": "medium",
        "message": "Budget range could indicate different trip tiers."
    },
    "traveler_count_conflict": {
        "action": "ASK_FOLLOWUP",
        "priority": "critical",
        "message": "Conflicting traveler counts. Please confirm."
    },
    "origin_conflict": {
        "action": "ASK_FOLLOWUP",
        "priority": "critical",
        "message": "Multiple departure cities detected. Please clarify."
    },
}

# Exact field-name → contradiction type mapping (no substring heuristics)
CONTRADICTION_FIELD_MAP = {
    # Destination fields
    "destination_city": "destination_conflict",
    "destination": "destination_conflict",
    "dest_city": "destination_conflict",
    "selected_destinations": "destination_conflict",
    # Date fields
    "travel_dates": "date_conflict",
    "travel_dates_from": "date_conflict",
    "travel_dates_to": "date_conflict",
    "date_window": "date_conflict",
    "dates": "date_conflict",
    # Budget fields
    "budget_range": "budget_conflict",
    "budget": "budget_conflict",
    "budget_total": "budget_conflict",
    "budget_band": "budget_conflict",
    # Traveler count fields
    "traveler_count": "traveler_count_conflict",
    "num_travelers": "traveler_count_conflict",
    "group_size": "traveler_count_conflict",
    # Origin fields
    "origin_city": "origin_conflict",
    "origin": "origin_conflict",
    "departure_city": "origin_conflict",
    "departure": "origin_conflict",
}

def classify_contradiction(field_name: str) -> str:
    """Map a field name to a contradiction type using exact alias matching."""
    ctype = CONTRADICTION_FIELD_MAP.get(field_name)
    if ctype:
        return ctype
    return "general_conflict"

def get_contradiction_action(contradiction_type: str) -> Dict[str, Any]:
    """Get the action for a contradiction type. Defaults to ASK_FOLLOWUP."""
    return CONTRADICTION_HANDLING.get(
        contradiction_type,
        {"action": "ASK_FOLLOWUP", "priority": "medium", "message": "Conflicting values detected."}
    )

## Confidence Scoring

Authority-weighted confidence calculation per the contract.

In [ ]:
# =============================================================================
# SECTION 3: CONFIDENCE SCORING
# =============================================================================

def calculate_confidence(packet: CanonicalPacket) -> float:
    """
    Weighted confidence based on:
    - Fact confidence (authority-weighted)
    - Hypothesis confidence (discounted to 0.5x)
    - Unknown penalties (0.1 per unknown)
    """
    # Authority-weighted fact confidence
    fact_weight = 0.0
    fact_count = 0
    for slot in packet.facts.values():
        auth_weight = AUTHORITY_WEIGHTS.get(slot.authority_level, 0.0)
        fact_weight += slot.confidence * auth_weight
        fact_count += 1

    if fact_count > 0:
        fact_weight /= fact_count  # Normalize

    # Hypothesis confidence (discounted)
    hypothesis_weight = 0.0
    hyp_count = 0
    for slot in packet.hypotheses.values():
        hypothesis_weight += slot.confidence * 0.5  # Discount hypotheses
        hyp_count += 1

    if hyp_count > 0:
        hypothesis_weight /= hyp_count

    # Unknown penalty
    unknown_penalty = len(packet.unknowns) * 0.1

    # Combine: facts are primary, hypotheses add a little, unknowns subtract
    raw = fact_weight + (hypothesis_weight * 0.2) - unknown_penalty

    return min(1.0, max(0.0, raw))


## Contradiction Detection

Detects conflicts between values in the packet.

In [ ]:
# =============================================================================
# SECTION 4: CONTRADICTION DETECTOR
# =============================================================================

def detect_contradictions(packet: CanonicalPacket) -> List[Dict[str, Any]]:
    """
    Find contradictions within the packet:
    1. Pre-existing contradictions from the packet
    2. Cross-layer contradictions (fact vs fact with different values for same semantic field)
    """
    contradictions = list(packet.contradictions)  # Start with existing

    # Check for cross-source contradictions in facts
    # If a single slot has evidence from multiple envelopes with different excerpts
    for field_name, slot in packet.facts.items():
        if len(slot.evidence_refs) >= 2:
            envelopes = set(ref.envelope_id for ref in slot.evidence_refs)
            if len(envelopes) >= 2:
                # Multiple sources — check if excerpts differ significantly
                excerpts = [ref.excerpt for ref in slot.evidence_refs]
                if len(set(excerpts)) > 1:
                    contradictions.append({
                        "field_name": field_name,
                        "type": "multi_source_conflict",
                        "values": excerpts,
                        "sources": list(envelopes),
                    })

    return contradictions


## Decision State Machine

Core logic flow per the contract:
```
1. Load CanonicalPacket
2. Check contradictions → if critical, STOP_NEEDS_REVIEW
3. Identify hard blockers for current stage
4. Identify soft blockers for current stage
5. Calculate overall confidence score
6. Determine decision state
7. Generate ordered follow-up questions
8. Return decision packet
```

In [ ]:
# =============================================================================
# SECTION 5: DECISION STATE MACHINE
# =============================================================================

DECISION_STATES = [
    "ASK_FOLLOWUP",
    "PROCEED_INTERNAL_DRAFT",
    "PROCEED_TRAVELER_SAFE",
    "BRANCH_OPTIONS",
    "STOP_NEEDS_REVIEW",
]

def evaluate_decision_state(
    packet: CanonicalPacket,
    hard_blockers: List[str],
    soft_blockers: List[str],
    contradictions: List[Dict[str, Any]],
    confidence: float,
    confidence_threshold: float = 0.6,
) -> Tuple[str, List[Dict[str, Any]], List[Dict[str, Any]], Dict[str, Any]]:
    """
    Determine the decision state and generate follow-up questions.

    Returns:
        (decision_state, follow_up_questions, branch_options, rationale)
    """
    follow_up_questions = []
    branch_options = []
    rationale = {
        "hard_blockers": hard_blockers,
        "soft_blockers": soft_blockers,
        "contradictions": [c["field_name"] for c in contradictions],
        "confidence": round(confidence, 3),
        "threshold": confidence_threshold,
    }

    # Step 2: Check contradictions → if critical, STOP_NEEDS_REVIEW
    critical_contradictions = []
    for c in contradictions:
        ctype = classify_contradiction(c["field_name"])
        action = get_contradiction_action(ctype)
        if action["priority"] == "critical":
            critical_contradictions.append({**c, "action": action, "type": ctype})

    if critical_contradictions:
        # Generate questions for critical contradictions
        for cc in critical_contradictions:
            follow_up_questions.append({
                "field_name": cc["field_name"],
                "question": cc["action"]["message"],
                "priority": "critical",
                "can_infer": False,
                "inference_confidence": 0.0,
            })

        # date_conflict → STOP_NEEDS_REVIEW; others → ASK_FOLLOWUP
        has_date_conflict = any(
            cc["type"] == "date_conflict" for cc in critical_contradictions
        )
        if has_date_conflict:
            return (
                "STOP_NEEDS_REVIEW",
                follow_up_questions,
                branch_options,
                {**rationale, "reason": "Critical date contradiction detected."},
            )
        else:
            return (
                "ASK_FOLLOWUP",
                follow_up_questions,
                branch_options,
                {**rationale, "reason": "Critical contradictions require clarification."},
            )

    # Step 3-4: Check hard blockers
    if hard_blockers:
        for blocker in hard_blockers:
            slot = resolve_field(packet, blocker)
            can_infer = False
            inference_conf = 0.0
            suggested = []

            # Check if a hypothesis exists (can't fill blocker but can suggest)
            hyp_slot = None
            if blocker in packet.hypotheses:
                hyp_slot = packet.hypotheses[blocker]
            else:
                for alias in FIELD_ALIASES.get(blocker, []):
                    if alias in packet.hypotheses:
                        hyp_slot = packet.hypotheses[alias]
                        break

            if hyp_slot:
                can_infer = True
                inference_conf = round(hyp_slot.confidence * 0.3, 2)  # Heavily discounted
                suggested = [hyp_slot.value] if hyp_slot.value else []

            follow_up_questions.append({
                "field_name": blocker,
                "question": _generate_question(blocker),
                "priority": "critical",
                "can_infer": can_infer,
                "inference_confidence": inference_conf,
                "suggested_values": suggested,
            })

        return (
            "ASK_FOLLOWUP",
            follow_up_questions,
            branch_options,
            {**rationale, "reason": f"{len(hard_blockers)} hard blocker(s) unresolved."},
        )

    # Step 5: Check for budget-level contradictions that suggest branching
    budget_contradictions = [
        c for c in contradictions
        if classify_contradiction(c["field_name"]) == "budget_conflict"
    ]
    if budget_contradictions and not hard_blockers:
        branch_options.append({
            "label": "Budget-tier options",
            "description": "Different budget interpretations suggest different trip tiers.",
            "contradictions": budget_contradictions,
        })
        return (
            "BRANCH_OPTIONS",
            follow_up_questions,
            branch_options,
            {**rationale, "reason": "Multiple valid budget paths exist."},
        )

    # Step 6: Check confidence threshold
    if confidence >= confidence_threshold and not soft_blockers:
        return (
            "PROCEED_TRAVELER_SAFE",
            follow_up_questions,
            branch_options,
            {**rationale, "reason": "All critical fields filled, confidence above threshold."},
        )

    # Soft blockers only → PROCEED_INTERNAL_DRAFT
    if soft_blockers:
        for blocker in soft_blockers:
            slot = resolve_field(packet, blocker)
            can_infer = slot is not None and is_hypothesis(slot.authority_level)
            inference_conf = round(slot.confidence * 0.5, 2) if slot else 0.0
            suggested = [slot.value] if slot and slot.value else []

            follow_up_questions.append({
                "field_name": blocker,
                "question": _generate_question(blocker),
                "priority": "high" if not can_infer else "medium",
                "can_infer": can_infer,
                "inference_confidence": inference_conf,
                "suggested_values": suggested,
            })

        return (
            "PROCEED_INTERNAL_DRAFT",
            follow_up_questions,
            branch_options,
            {**rationale, "reason": "Soft blockers remain — internal draft only."},
        )

    # Fallback: if confidence is below threshold but no blockers
    if confidence < confidence_threshold:
        return (
            "PROCEED_INTERNAL_DRAFT",
            follow_up_questions,
            branch_options,
            {**rationale, "reason": f"Confidence {confidence:.2f} below threshold {confidence_threshold}."},
        )

    return (
        "PROCEED_TRAVELER_SAFE",
        follow_up_questions,
        branch_options,
        {**rationale, "reason": "All checks passed."},
    )


def _generate_question(field_name: str) -> str:
    """Generate a human-readable question for a missing field."""
    questions = {
        "destination_city": "Where would you like to go?",
        "origin_city": "Which city will you be departing from?",
        "travel_dates": "When are you planning to travel? Are the dates flexible?",
        "traveler_count": "How many people will be traveling?",
        "budget_range": "What's your approximate budget for this trip?",
        "trip_purpose": "What's the main purpose of this trip? (leisure, business, pilgrimage, etc.)",
        "traveler_preferences": "Any specific preferences or must-haves for this trip?",
        "selected_destinations": "Which destinations from the shortlist interest you most?",
        "selected_itinerary": "Which itinerary option do you prefer?",
        "activity_preferences": "What kind of activities are you interested in?",
        "accommodation_type": "What type of accommodation do you prefer? (hotel, resort, homestay, etc.)",
        "special_requests": "Any special requests or requirements we should know about?",
        "dietary_restrictions": "Any dietary restrictions or food preferences?",
        "traveler_details": "We'll need full legal names and passport details for booking. Can you provide those?",
        "payment_method": "How would you like to handle payment? (card, transfer, etc.)",
    }
    return questions.get(field_name, f"Can you provide details for: {field_name}?")


## Main Pipeline: `run_gap_and_decision`

Single entry point that executes the full contract logic.

In [ ]:
# =============================================================================
# SECTION 6: MAIN PIPELINE
# =============================================================================

@dataclass
class DecisionResult:
    """The complete output of the gap and decision pipeline."""
    packet_id: str
    current_stage: str
    hard_blockers: List[str]
    soft_blockers: List[str]
    contradictions: List[Dict[str, Any]]
    decision_state: str
    follow_up_questions: List[Dict[str, Any]]
    branch_options: List[Dict[str, Any]]
    rationale: Dict[str, Any]
    confidence_score: float

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)


def run_gap_and_decision(
    packet: CanonicalPacket,
    current_stage: Optional[str] = None,
    mvb_config: Optional[Dict] = None,
    confidence_threshold: float = 0.6,
) -> DecisionResult:
    """
    Full pipeline: CanonicalPacket → DecisionState.

    Steps:
    1. Load CanonicalPacket
    2. Check contradictions → if critical, STOP_NEEDS_REVIEW
    3. Identify hard blockers for current stage
    4. Identify soft blockers for current stage
    5. Calculate overall confidence score
    6. Determine decision state
    7. Generate ordered follow-up questions
    8. Return decision packet
    """
    stage = current_stage or packet.stage
    mvb = mvb_config or MVB_BY_STAGE.get(stage, MVB_BY_STAGE["discovery"])

    # Step 1: Detect contradictions
    contradictions = detect_contradictions(packet)

    # Step 2-3: Identify hard blockers
    # Only facts and derived signals can resolve hard blockers (NOT hypotheses)
    hard_blockers = []
    for field_name in mvb["hard_blockers"]:
        slot = resolve_field(packet, field_name)
        if not field_fills_blocker(slot):
            hard_blockers.append(field_name)

    # Step 4: Identify soft blockers
    soft_blockers = []
    for field_name in mvb["soft_blockers"]:
        slot = resolve_field(packet, field_name)
        if slot is None:
            soft_blockers.append(field_name)
        # Soft blockers can be satisfied by hypotheses too

    # Step 5: Calculate confidence
    confidence = calculate_confidence(packet)

    # Step 6-7: Evaluate decision state and generate questions
    decision_state, follow_up, branches, rationale = evaluate_decision_state(
        packet=packet,
        hard_blockers=hard_blockers,
        soft_blockers=soft_blockers,
        contradictions=contradictions,
        confidence=confidence,
        confidence_threshold=confidence_threshold,
    )

    return DecisionResult(
        packet_id=packet.packet_id,
        current_stage=stage,
        hard_blockers=hard_blockers,
        soft_blockers=soft_blockers,
        contradictions=contradictions,
        decision_state=decision_state,
        follow_up_questions=follow_up,
        branch_options=branches,
        rationale=rationale,
        confidence_score=round(confidence, 3),
    )


## Worked Examples

### Test 1: Empty packet → ASK_FOLLOWUP with all hard blockers

In [ ]:
# Test 1: Empty packet → ASK_FOLLOWUP
empty_packet = CanonicalPacket(
    packet_id="pkt_empty",
    created_at=datetime.now().isoformat(),
    last_updated=datetime.now().isoformat(),
    stage="discovery",
)

result1 = run_gap_and_decision(empty_packet, current_stage="discovery")
print(f"Decision: {result1.decision_state}")
print(f"Hard blockers: {result1.hard_blockers}")
print(f"Soft blockers: {result1.soft_blockers}")
print(f"Confidence: {result1.confidence_score}")
print(f"Questions: {len(result1.follow_up_questions)}")
for q in result1.follow_up_questions:
    print(f"  [{q['priority']}] {q['question']}")
assert result1.decision_state == "ASK_FOLLOWUP", f"Expected ASK_FOLLOWUP, got {result1.decision_state}"
assert len(result1.hard_blockers) == 4, f"Expected 4 hard blockers, got {len(result1.hard_blockers)}"
print("PASS: Empty packet correctly returns ASK_FOLLOWUP with all hard blockers")


### Test 2: Complete packet → PROCEED_TRAVELER_SAFE

In [ ]:
# Test 2: Complete packet → PROCEED_TRAVELER_SAFE
# Must include ALL hard AND soft blockers to get PROCEED_TRAVELER_SAFE
complete_packet = CanonicalPacket(
    packet_id="pkt_complete",
    created_at=datetime.now().isoformat(),
    last_updated=datetime.now().isoformat(),
    facts={
        "destination_city": Slot(value="Singapore", confidence=0.95, authority_level="explicit_user"),
        "origin_city": Slot(value="Bangalore", confidence=0.95, authority_level="explicit_user"),
        "travel_dates": Slot(value="2026-03-15 to 2026-03-22", confidence=0.90, authority_level="explicit_user"),
        "traveler_count": Slot(value=5, confidence=0.95, authority_level="explicit_user"),
        "budget_range": Slot(value="250000", confidence=0.80, authority_level="explicit_owner"),
        "trip_purpose": Slot(value="family leisure", confidence=0.85, authority_level="explicit_user"),
        "traveler_preferences": Slot(value="relaxed pace, kid-friendly", confidence=0.80, authority_level="explicit_user"),
    },
    stage="discovery",
)

result2 = run_gap_and_decision(complete_packet, current_stage="discovery")
print(f"Decision: {result2.decision_state}")
print(f"Hard blockers: {result2.hard_blockers}")
print(f"Soft blockers: {result2.soft_blockers}")
print(f"Confidence: {result2.confidence_score}")
assert result2.decision_state == "PROCEED_TRAVELER_SAFE", f"Expected PROCEED_TRAVELER_SAFE, got {result2.decision_state}"
assert len(result2.hard_blockers) == 0, f"Expected 0 hard blockers, got {result2.hard_blockers}"
assert len(result2.soft_blockers) == 0, f"Expected 0 soft blockers, got {result2.soft_blockers}"
print("PASS: Complete packet correctly returns PROCEED_TRAVELER_SAFE")

### Test 3: Packet with hypothesis only → ASK_FOLLOWUP (hypothesis doesn't fill blocker)

In [ ]:
# Test 3: Hypothesis-only packet → ASK_FOLLOWUP
# Hypotheses do NOT fill blockers per contract
hypothesis_packet = CanonicalPacket(
    packet_id="pkt_hypothesis",
    created_at=datetime.now().isoformat(),
    last_updated=datetime.now().isoformat(),
    facts={
        "origin_city": Slot(value="Bangalore", confidence=0.90, authority_level="explicit_owner"),
    },
    hypotheses={
        "destination_city": Slot(value="Singapore", confidence=0.50, authority_level="soft_hypothesis"),
        "travel_dates": Slot(value="March 2026", confidence=0.40, authority_level="soft_hypothesis"),
        "traveler_count": Slot(value=4, confidence=0.45, authority_level="soft_hypothesis"),
    },
    stage="discovery",
)

result3 = run_gap_and_decision(hypothesis_packet, current_stage="discovery")
print(f"Decision: {result3.decision_state}")
print(f"Hard blockers: {result3.hard_blockers}")
print(f"Confidence: {result3.confidence_score}")
# destination_city, travel_dates, traveler_count should still be blockers
# (hypotheses don't fill them)
assert result3.decision_state == "ASK_FOLLOWUP", f"Expected ASK_FOLLOWUP, got {result3.decision_state}"
assert "destination_city" in result3.hard_blockers, "destination_city should still be a blocker"
assert "traveler_count" in result3.hard_blockers, "traveler_count should still be a blocker"
assert "travel_dates" in result3.hard_blockers, "travel_dates should still be a blocker"
# Questions should note that hypotheses exist for these fields
for q in result3.follow_up_questions:
    print(f"  [{q['priority']}] {q['question']} (can_infer={q['can_infer']}, conf={q['inference_confidence']})")
print("PASS: Hypothesis-only packet correctly returns ASK_FOLLOWUP — hypotheses don't fill blockers")


### Test 3b: Derived signal fills a hard blocker (per contract rule)

In [ ]:
# Test 3b: Derived signal → fills a hard blocker
# Contract: "Derived signals CAN fill blockers — if authority is derived_signal"
derived_signal_packet = CanonicalPacket(
    packet_id="pkt_derived",
    created_at=datetime.now().isoformat(),
    last_updated=datetime.now().isoformat(),
    facts={
        "origin_city": Slot(value="Bangalore", confidence=0.90, authority_level="explicit_owner"),
        "traveler_count": Slot(value=5, confidence=0.95, authority_level="explicit_owner"),
        "travel_dates": Slot(value="2026-03-15 to 2026-03-22", confidence=0.90, authority_level="explicit_user"),
    },
    derived_signals={
        # Derived from itinerary notes — fills the destination blocker
        "destination_city": Slot(
            value="Singapore",
            confidence=0.75,
            authority_level="derived_signal",
            extraction_mode="inferred",
            evidence_refs=[
                EvidenceRef(
                    ref_id="ref_derived",
                    envelope_id="env_notes",
                    evidence_type="derived",
                    excerpt="Inferred from flight route + hotel mentions"
                )
            ]
        ),
    },
    stage="discovery",
)

result3b = run_gap_and_decision(derived_signal_packet, current_stage="discovery")
print(f"Decision: {result3b.decision_state}")
print(f"Hard blockers: {result3b.hard_blockers}")
print(f"Soft blockers: {result3b.soft_blockers}")
print(f"Confidence: {result3b.confidence_score}")

# destination_city should NOT be a blocker — derived_signal fills it
assert "destination_city" not in result3b.hard_blockers, \
    f"derived_signal should fill destination_city blocker, but it is still blocked: {result3b.hard_blockers}"
# Hard blockers: none (all 4 filled: 3 facts + 1 derived_signal)
assert len(result3b.hard_blockers) == 0, \
    f"Expected 0 hard blockers, got {result3b.hard_blockers}"
# Soft blockers should still exist
assert len(result3b.soft_blockers) > 0, "Should still have soft blockers"
print("PASS: Derived signal correctly fills a hard blocker")

### Test 4: Packet with contradiction → STOP_NEEDS_REVIEW or ASK_FOLLOWUP per policy

In [ ]:
# Test 4a: Date contradiction → STOP_NEEDS_REVIEW (critical per policy)
date_contradiction_packet = CanonicalPacket(
    packet_id="pkt_date_conflict",
    created_at=datetime.now().isoformat(),
    last_updated=datetime.now().isoformat(),
    facts={
        "destination_city": Slot(value="Singapore", confidence=0.90, authority_level="explicit_user"),
        "origin_city": Slot(value="Bangalore", confidence=0.90, authority_level="explicit_user"),
        "travel_dates": Slot(
            value=["2026-03-15", "2026-04-01"],
            confidence=0.70,
            authority_level="explicit_owner",
            evidence_refs=[
                EvidenceRef(ref_id="ref1", envelope_id="env1", evidence_type="text_span", excerpt="March 15"),
                EvidenceRef(ref_id="ref2", envelope_id="env2", evidence_type="text_span", excerpt="April 1"),
            ],
        ),
        "traveler_count": Slot(value=3, confidence=0.90, authority_level="explicit_user"),
    },
    contradictions=[
        {"field_name": "travel_dates", "values": ["2026-03-15", "2026-04-01"], "sources": ["env1", "env2"]},
    ],
    stage="discovery",
)

result4a = run_gap_and_decision(date_contradiction_packet, current_stage="discovery")
print(f"Decision (date conflict): {result4a.decision_state}")
assert result4a.decision_state == "STOP_NEEDS_REVIEW", f"Expected STOP_NEEDS_REVIEW, got {result4a.decision_state}"
print("PASS: Date contradiction correctly returns STOP_NEEDS_REVIEW")

# Test 4b: Budget contradiction → BRANCH_OPTIONS (medium priority per policy)
budget_contradiction_packet = CanonicalPacket(
    packet_id="pkt_budget_conflict",
    created_at=datetime.now().isoformat(),
    last_updated=datetime.now().isoformat(),
    facts={
        "destination_city": Slot(value="Singapore", confidence=0.90, authority_level="explicit_user"),
        "origin_city": Slot(value="Bangalore", confidence=0.90, authority_level="explicit_user"),
        "travel_dates": Slot(value="2026-03-15 to 2026-03-22", confidence=0.90, authority_level="explicit_user"),
        "traveler_count": Slot(value=3, confidence=0.90, authority_level="explicit_user"),
        "budget_range": Slot(
            value=["budget", "premium"],
            confidence=0.60,
            authority_level="explicit_owner",
            evidence_refs=[
                EvidenceRef(ref_id="ref3", envelope_id="env3", evidence_type="text_span", excerpt="budget trip"),
                EvidenceRef(ref_id="ref4", envelope_id="env4", evidence_type="text_span", excerpt="wants premium hotels"),
            ],
        ),
    },
    contradictions=[
        {"field_name": "budget_range", "values": ["budget", "premium"], "sources": ["env3", "env4"]},
    ],
    stage="discovery",
)

result4b = run_gap_and_decision(budget_contradiction_packet, current_stage="discovery")
print(f"Decision (budget conflict): {result4b.decision_state}")
assert result4b.decision_state == "BRANCH_OPTIONS", f"Expected BRANCH_OPTIONS, got {result4b.decision_state}"
assert len(result4b.branch_options) > 0, "Should have branch options"
print("PASS: Budget contradiction correctly returns BRANCH_OPTIONS")


### Test 5: Packet with soft blockers only → PROCEED_INTERNAL_DRAFT

In [ ]:
# Test 5: Soft blockers only → PROCEED_INTERNAL_DRAFT
soft_blocker_packet = CanonicalPacket(
    packet_id="pkt_soft",
    created_at=datetime.now().isoformat(),
    last_updated=datetime.now().isoformat(),
    facts={
        "destination_city": Slot(value="Singapore", confidence=0.90, authority_level="explicit_user"),
        "origin_city": Slot(value="Bangalore", confidence=0.90, authority_level="explicit_user"),
        "travel_dates": Slot(value="2026-03-15 to 2026-03-22", confidence=0.90, authority_level="explicit_user"),
        "traveler_count": Slot(value=3, confidence=0.90, authority_level="explicit_user"),
    },
    # soft_blockers for discovery: budget_range, trip_purpose, traveler_preferences
    stage="discovery",
)

result5 = run_gap_and_decision(soft_blocker_packet, current_stage="discovery")
print(f"Decision: {result5.decision_state}")
print(f"Hard blockers: {result5.hard_blockers}")
print(f"Soft blockers: {result5.soft_blockers}")
assert result5.decision_state == "PROCEED_INTERNAL_DRAFT", f"Expected PROCEED_INTERNAL_DRAFT, got {result5.decision_state}"
assert len(result5.hard_blockers) == 0, "No hard blockers"
assert len(result5.soft_blockers) > 0, "Should have soft blockers"
print("PASS: Soft blockers only correctly returns PROCEED_INTERNAL_DRAFT")


## Full Output Example

JSON representation of a complete decision result.

In [ ]:
# Print the complete decision result as JSON
print(json.dumps(result2.to_dict(), indent=2, default=str))


## Summary

| Test | Packet State | Expected Decision | Result |
|------|-------------|-------------------|--------|
| 1 | Empty packet | ASK_FOLLOWUP (4 hard blockers) | PASS |
| 2 | Complete facts | PROCEED_TRAVELER_SAFE | PASS |
| 3 | Hypotheses only | ASK_FOLLOWUP (hypotheses don't fill) | PASS |
| 3b | Derived signal | Blocker filled by derived_signal | PASS |
| 4a | Date contradiction | STOP_NEEDS_REVIEW | PASS |
| 4b | Budget contradiction | BRANCH_OPTIONS | PASS |
| 5 | Soft blockers only | PROCEED_INTERNAL_DRAFT | PASS |

**Key contract rules enforced:**
1. Hypotheses do NOT fill hard blockers AND do NOT boost confidence (weight = 0.0)
2. Derived signals CAN fill hard blockers (if authority is `derived_signal`)
3. Date contradictions → STOP_NEEDS_REVIEW (critical)
4. Budget contradictions → BRANCH_OPTIONS (medium)
5. Soft blockers → PROCEED_INTERNAL_DRAFT
6. Questions ordered by impact: critical → high → medium → low
7. Contradiction classification uses exact field-name/alias mapping (no substring heuristics)